In [15]:
import ee
import numpy as np
import pandas as pd
import geopandas
from pyproj import Proj, Transformer
import os
import time
from itertools import batched
from pathlib import Path

In [ ]:
 ee.Authenticate(auth_mode='localhost', force=True)

In [16]:
ee.Initialize(project='geewildfires')

In [17]:
def coordinates_from_geodataframe(df):
    transformer = Transformer.from_crs(df['geometry'].crs, "EPSG:4326", always_xy=True)
    df["lon1"], df["lat1"] = transformer.transform(df["geometry"].x, df["geometry"].y)
    df = df[["FIRE_ID", "IG_DATE", "Occurrence", "point_orde", "lon1", "lat1"]]
    return df

def extract_features_from_dataframe_points(df):
    features=[]
    for index, row in df.iterrows():
        point_geometry = ee.Geometry.Point(row['lon1'], row['lat1'])
        point_properties = {'fire_id': row["FIRE_ID"], 'date': str(row["IG_DATE"]), 'occur_id':  str(row["Occurrence"]), "point_id": str(row["point_orde"])}
        point_feature = ee.Feature(point_geometry, point_properties)
        features.append(point_feature)
        
    feature_collection = ee.FeatureCollection(features)
    return feature_collection

def year_extract_features_from_dataframe_points(df):
    features=[]
    for index, row in df.iterrows():
        point_geometry = ee.Geometry.Point(row['lon1'], row['lat1'])
        point_properties = {'fire_id': row["FIRE_ID"], 'date': str(row["IG_DATE"]), 'occur_id':  str(row["Occurrence"]), "point_id": str(row["point_orde"])}
        point_feature = ee.Feature(point_geometry, point_properties)
        features.append(point_feature)
        
    feature_collection = ee.FeatureCollection(features)
    return feature_collection

def get_terrain_collection(collection_name):
    terrain = ee.ImageCollection(collection_name)
    return terrain

def year_get_image_for_date(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 730
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if filtered_test.limit(1).size().getInfo():
            image_found = True
        elif date_counter > 2000:
            return None
        else:
            date_counter += 365
    return selected_collection, date_counter

def get_image_for_date(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 17
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if date_counter > 180:
            return None
        elif filtered_test is None:
            date_counter += 17
        else:
            image_found = True
    return selected_collection, date_counter

def get_image_for_date_final(collection_name, date_string, feature):
    date = str(date_string[:10])
    date_counter = 20
    image_found = False
    while not image_found:
        date_start = str((pd.Timestamp(date) - pd.Timedelta(days=date_counter)).date())
        selected_collection = ee.ImageCollection(collection_name).filterDate(date_start, date)
        filtered_test = selected_collection.filterBounds(feature.geometry())
        if filtered_test.size().getInfo() > 0:
            image_found = True
        elif date_counter > 180:
            return None
        else:
            date_counter += 20
    return selected_collection, date_counter

def merge_feature_collection_list(feature_collection_list):
    result_feature_collection = feature_collection_list[0]
    for feature_collection in feature_collection_list[1:]:
        result_feature_collection = result_feature_collection.merge(feature_collection)
    return result_feature_collection

def set_target_projection(target_projection):
    if isinstance(target_projection, ee.Projection):
        target_proj = target_projection
    else:
        target_proj = ee.Projection(f'EPSG:{target_projection}')
    return target_proj

def raster_reduce(image, feature, scale=30):
    feature_sample = image.reduceRegions(
        collection = feature,
        reducer=ee.Reducer.first(),
        scale = scale,
    )
    return feature_sample

def terrain_reduce(image, feature, scale=90):
    terrain = ee.Terrain.products(image)
    feature_sample = terrain.reduceRegions(
        collection = feature,
        reducer=ee.Reducer.first(),
        scale = scale,
    )
    return feature_sample

def apply_raster_extraction(collection_name, measures, feature, target_projection):
    target_proj = set_target_projection(target_projection)
    filtered = collection_name.filterBounds(feature.geometry())
    results = filtered.select(measures).filterBounds(feature.geometry()).mosaic().setDefaultProjection(target_proj)
    return results

def export_to_drive(tasks, raster_sampled, description, i, request_id):
    task = ee.batch.Export.table.toDrive(
        collection=raster_sampled,
        description=f'{description}_{i}_{request_id}',
        folder='wildfire_validation',
        fileNamePrefix=f'{description}_{i}_{request_id}',
        fileFormat='CSV'
    )
    task.start()
    tasks.append(task)

In [18]:
def get_dated_point_data(df, collection, measure_list, description, dated=True, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0
        
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('IG_DATE')]
    total_batches = len(date_batches)

    for date, date_group in date_batches:
        i += 1
        try:
            point_feature_collection = extract_features_from_dataframe_points(date_group)
            date_str = date

            result_image = get_image_for_date_final(collection, date_str, point_feature_collection)
            if result_image is None:
                continue
            test_collection, days_back = result_image

            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, 4326)
            raster_reduced = raster_reduce(raster_extracted, point_feature_collection, scale=30)

            batch_all_features.append(raster_reduced)
        
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f"{len(batch_all_features)} feature collections out of {total_batches} date batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 2):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 2
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 2000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0
        

In [19]:
def get_fixed_point_data(df, collection, measure_list, description, dated=False, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0
    
    fires_df_coords['year'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.year
    fires_df_coords['month'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.month
    fires_df_coords['year_month'] = fires_df_coords['year'].astype(str) + '_' + fires_df_coords['month'].astype(str)
    
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('year_month')]
    total_batches = len(date_batches)
    
    for date, date_group in date_batches:
        i += 1
        try:
            point_feature_collection = extract_features_from_dataframe_points(date_group)
            test_collection = get_terrain_collection(collection)
            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, test_collection.first().projection())
            raster_reduced = terrain_reduce(raster_extracted, point_feature_collection, scale=90)

            batch_all_features.append(raster_reduced)
            
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f"{len(batch_all_features)} feature collections out of {total_batches} batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 1):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 1
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 1000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0

In [20]:
def get_year_point_data(df, collection, measure_list, description, dated=True, request_id=1):
    fires_df_coords = coordinates_from_geodataframe(df)
    extracted_feature_list = []
    tasks = []
    batch_all_features = []
    i = 0
    request_count = 0
    request_old = 0

    fires_df_coords['year'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.year
    fires_df_coords['month'] = pd.to_datetime(fires_df_coords['IG_DATE']).dt.month
    fires_df_coords['month'] = fires_df_coords['month'].astype(str).str.zfill(2)
    fires_df_coords['year_month'] = fires_df_coords['year'].astype(str) + '-' + fires_df_coords['month'].astype(str)
    
    date_batches = [(date, date_group) for date, date_group in fires_df_coords.groupby('year_month')]
    total_batches = len(date_batches)

    for date, date_group in date_batches:
        i += 1
        date_pop = str(date)
        try:
            point_feature_collection = year_extract_features_from_dataframe_points(date_group)
            date_str = set_five_year_date_pop(date_pop)

            result_image = year_get_image_for_date(collection, date_str, point_feature_collection)
            if result_image is None:
                continue
            test_collection, days_back = result_image
            raster_extracted = apply_raster_extraction(test_collection, measure_list, point_feature_collection, 4326)
            raster_reduced = raster_reduce(raster_extracted, point_feature_collection, scale=30)

            batch_all_features.append(raster_reduced)
        
        except:
            # future improvement: log failed FIRE_IDs here
            continue
    
    print(f" {len(batch_all_features)} feature collections out of {total_batches} batches.")
    batch_check = 0

    for feature_collection_group in batched(batch_all_features, 1):
        request_id += 1
        batch_check += 1
        request_old = request_count + 1
        request_count += 1
        raster_batched = merge_feature_collection_list(feature_collection_group)
        print(f"Getting dates from {request_old} to {request_count}")
        export_to_drive(tasks, raster_batched, description, i, request_id)
        if batch_check > 2000:
            time.sleep(300)
            print("Waiting for queue to clear")
            batch_check = 0

In [21]:
landsat7_collection = "LANDSAT/LE07/C02/T1_L2"
landsat7_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
landsat5_collection = "LANDSAT/LT05/C02/T1_L2"
landsat5_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
landsat8_collection = "LANDSAT/LC08/C02/T1_L2"
landsat8_measures = ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']

elevation_collection = 'COPERNICUS/DEM/GLO30'
elevation_measures = ['DEM']

weather_metrics = ['tmmx', 'tmmn', 'pr', 'rmax', 'rmin', 'vs', 'th', 'erc', 'bi', 'fm100', 'fm1000', 'eto', 'vpd']
weather_collection = 'IDAHO_EPSCOR/GRIDMET'

pop_density_collection = 'CIESIN/GPWv411/GPW_Population_Density'
pop_density_metrics = 'population_density'

land_cover_collection = "JRC/GHSL/P2023A/GHS_SMOD_V2-0"
land_cover_metrics = "smod_code"

# remember the notebook needs to be Trusted for the calls to work properly

In [22]:
def download_landsat_data(df, r_id=1):
    # added a couple of months to each of these, just because program needs to look back for images
    fires_ldf_8 = df[df['IG_DATE'] > '2013-05-17']
    fires_ldf_7 = df.loc[(df['IG_DATE'] > '1999-07-28') & (df['IG_DATE'] < '2013-05-17')]
    fires_ldf_5 = df[df['IG_DATE'] < '1999-07-29']
    fires_df = df
    
    if len(fires_ldf_8) >= 0:
        landsat_8_data = get_dated_point_data(fires_ldf_8, landsat8_collection, ['SR_B2','SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'], "landsat8", request_id=r_id)
    if len(fires_ldf_7) >= 0:
        landsat_7_data = get_dated_point_data(fires_ldf_7, landsat7_collection, ['SR_B2','SR_B3','SR_B4', 'SR_B5', 'SR_B7'], "landsat7", request_id=r_id)
    if len(fires_ldf_5) >= 0:
        landsat_5_data = get_dated_point_data(fires_ldf_5, landsat5_collection, ['SR_B2','SR_B3','SR_B4', 'SR_B5', 'SR_B7'], "landsat5", request_id=r_id)
    
def download_gridmet_data(df, r_id = 1):
    gridmet_data = get_dated_point_data(df, weather_collection, weather_metrics, "weather", request_id=r_id)
    
def download_elevation_data(df, r_id=1):
    terrain_data = get_fixed_point_data(df, elevation_collection, ['DEM'], "elevation", dated=False, request_id=r_id)
    
def download_pop_density_data(df, r_id = 1):
    pop_density_data = get_year_point_data(df, pop_density_collection, pop_density_metrics, "pop_density", request_id=r_id)

def download_landcover_data(df, r_id = 1):
    land_cover_data = get_year_point_data(df, land_cover_collection, land_cover_metrics, "land_cover", request_id=r_id)

In [23]:
def set_five_year_date(date):
    new_date = '2020-12-31'
    year = pd.to_datetime(date).year
    if year >= 2015:
        new_date = '2015-12-31'
    elif year >= 2010:
        new_date = '2010-12-31'
    elif year >= 2005:
        new_date = '2005-12-31'
    elif year >= 2000:
        new_date = '2000-12-31'
    elif year >= 1995:
        new_date = '1995-12-31'
    elif year >= 1990:
        new_date = '1990-12-31'
    elif year >= 1985:
        new_date = '1985-12-31'
    elif year >= 1980:
        new_date = '1980-12-31'
        
    return new_date

def set_five_year_date_pop(date):
    new_date = '2020-12-31'
    year = pd.to_datetime(date).year
    if year >= 2015:
        new_date = '2015-12-31'
    elif year >= 2010:
        new_date = '2010-12-31'
    elif year >= 2005:
        new_date = '2005-12-31'
    elif year >= 1970:
        new_date = '2000-12-31'
        
    return new_date

In [24]:
def load_gee_from_folder(folder, data_name):
    import os
    dfs = []
    for filename in os.listdir(folder):
        if data_name in filename:
            file_r = pd.read_csv(os.path.join(folder, filename), engine='python')
            dfs.append(file_r)
    all_df = pd.concat(dfs, axis=0, ignore_index=True)
    return all_df

In [25]:
fires_ldf = geopandas.read_file(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartTwoOutput3.shp")
fires_ldf = fires_ldf
print(len(fires_ldf))

2387


In [26]:
fires_ldf_trunc = fires_ldf.to_crs(5070)
fires_ldf_trunc

,FIRE_ID,IG_DATE,FIRE_TYPE,Occurrence,point_orde,geometry
0,CA3622612010420250902,2025-09-02,Wildfire,1,0,POINT (-2122095.131 1735488.284)
1,CA3622612010420250902,2025-09-02,Wildfire,1,1,POINT (-2122120.574 1735445.241)
2,CA3622612010420250902,2025-09-02,Wildfire,1,2,POINT (-2122146.018 1735402.199)
3,CA3622612010420250902,2025-09-02,Wildfire,1,3,POINT (-2122171.461 1735359.157)
4,CA3622612010420250902,2025-09-02,Wildfire,1,4,POINT (-2122196.904 1735316.115)
...,...,...,...,...,...,...
2382,NC3522508227420250302,2025-03-02,Wildfire,13,95,POINT (1237653.958 1441250.045)
2383,NC3522508227420250302,2025-03-02,Wildfire,13,96,POINT (1237703.949 1441249.119)
2384,NC3522508227420250302,2025-03-02,Wildfire,13,97,POINT (1237753.941 1441248.193)
2385,NC3522508227420250302,2025-03-02,Wildfire,13,98,POINT (1237803.932 1441247.267)


In [32]:
download_elevation_data(fires_ldf_trunc, r_id=1)

4 feature collections out of 4 batches.
Getting dates from 1 to 1
Getting dates from 2 to 2
Getting dates from 3 to 3
Getting dates from 4 to 4


In [28]:
download_gridmet_data(fires_ldf_trunc, r_id=1)

5 feature collections out of 5 date batches.
Getting dates from 1 to 2
Getting dates from 3 to 4
Getting dates from 5 to 6


In [29]:
download_landsat_data(fires_ldf_trunc, r_id=1)

5 feature collections out of 5 date batches.
Getting dates from 1 to 2
Getting dates from 3 to 4
Getting dates from 5 to 6
0 feature collections out of 0 date batches.
0 feature collections out of 0 date batches.


In [30]:
download_pop_density_data(fires_ldf_trunc, r_id=1)

 4 feature collections out of 4 batches.
Getting dates from 1 to 1
Getting dates from 2 to 2
Getting dates from 3 to 3
Getting dates from 4 to 4


In [31]:
download_landcover_data(fires_ldf_trunc, r_id=1)

 4 feature collections out of 4 batches.
Getting dates from 1 to 1
Getting dates from 2 to 2
Getting dates from 3 to 3
Getting dates from 4 to 4


In [ ]:
# reminder: run each of the downloads and wait for them to complete before continuing

In [22]:
import gc
gc.collect()

910

In [ ]:
# reminder: download everything from Google Drive into the local folder

In [33]:
folder = r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\GEE_Downloads\wildfire_validation"

topology_df = load_gee_from_folder(folder, 'elevation')
weather_df = load_gee_from_folder(folder, 'weather')
landsat8_df = load_gee_from_folder(folder, 'landsat8')
#landsat7_df = load_gee_from_folder(folder, 'landsat7')
#landsat5_df = load_gee_from_folder(folder, 'landsat5')
pop_density_df = load_gee_from_folder(folder, 'pop_density')
land_cover_df = load_gee_from_folder(folder, 'land_cover')
topology_df.head()

,system:index,DEM,aspect,date,fire_id,hillshade,occur_id,point_id,slope,.geo
0,0,531.721436,150,2025-01-29,VA3712108003520250129,150,9,0,20,"{""type"":""Point"",""coordinates"":[-80.04557007724..."
1,1,522.399597,231,2025-01-29,VA3712108003520250129,207,9,1,11,"{""type"":""Point"",""coordinates"":[-80.04539567321..."
2,2,522.399597,231,2025-01-29,VA3712108003520250129,207,9,2,11,"{""type"":""Point"",""coordinates"":[-80.04522126720..."
3,3,530.713928,226,2025-01-29,VA3712108003520250129,211,9,3,13,"{""type"":""Point"",""coordinates"":[-80.04504685921..."
4,4,530.713928,226,2025-01-29,VA3712108003520250129,211,9,4,13,"{""type"":""Point"",""coordinates"":[-80.04487244923..."


In [34]:
topology_df.columns

Index(['system:index', 'DEM', 'aspect', 'date', 'fire_id', 'hillshade',
       'occur_id', 'point_id', 'slope', '.geo'],
      dtype='str')

In [35]:
topology_df = topology_df.drop(['system:index'], axis=1)
weather_df = weather_df.drop(['system:index'], axis=1)
landsat8_df = landsat8_df.drop(['system:index'], axis=1)
#landsat7_df = landsat7_df.drop(['system:index'], axis=1)
#landsat5_df = landsat5_df.drop(['system:index'], axis=1)
pop_density_df= pop_density_df.drop(['system:index', 'date'], axis=1)
land_cover_df = land_cover_df.drop(['system:index', 'date'], axis=1)

In [36]:
concat_df = landsat8_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id']) #pd.concat([landsat8_df, landsat7_df, landsat5_df])

In [37]:
len(concat_df)

2387

In [38]:
concat_df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
Index: 2387 entries, 0 to 3583
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   SR_B2     2387 non-null   int64
 1   SR_B3     2387 non-null   int64
 2   SR_B4     2387 non-null   int64
 3   SR_B5     2387 non-null   int64
 4   SR_B7     2387 non-null   int64
 5   date      2387 non-null   str  
 6   fire_id   2387 non-null   str  
 7   occur_id  2387 non-null   int64
 8   point_id  2387 non-null   int64
 9   .geo      2387 non-null   str  
dtypes: int64(7), str(3)
memory usage: 205.1 KB


In [110]:
weather_df = weather_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id'])

In [111]:
len(pop_density_df)

2387

In [112]:
pop_density_df['pop_density'] = pop_density_df['first']
pop_density_df.drop(['first', '.geo'], axis=1, inplace=True)

pop_density_df.columns
len(pop_density_df)

2387

In [113]:
pop_density_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id'], inplace=True)
len(pop_density_df)

2387

In [114]:
pop_density_df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2387 entries, 0 to 2386
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   fire_id      2387 non-null   str    
 1   occur_id     2387 non-null   int64  
 2   point_id     2387 non-null   int64  
 3   pop_density  2387 non-null   float64
dtypes: float64(1), int64(2), str(1)
memory usage: 74.7 KB


In [115]:
print(land_cover_df.columns)
len(land_cover_df)

Index(['fire_id', 'first', 'occur_id', 'point_id', '.geo'], dtype='str')


2387

In [116]:
land_cover_df['u_class'] = land_cover_df['first']
land_cover_df.drop(['first', '.geo'], axis=1, inplace=True)

In [117]:
land_cover_df.drop_duplicates(subset=['fire_id', 'occur_id', 'point_id'], inplace=True)
len(land_cover_df)

2387

In [118]:
final_df = concat_df.merge(topology_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

2387

In [119]:
final_df = final_df.merge(weather_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

2387

In [120]:
final_df = final_df.merge(pop_density_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

2387

In [121]:
final_df = final_df.merge(land_cover_df, how='left', on=['fire_id', 'occur_id', 'point_id'])
len(final_df)

2387

In [122]:
final_df.columns

Index(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'date_x', 'fire_id',
       'occur_id', 'point_id', '.geo_x', 'DEM', 'aspect', 'date_y',
       'hillshade', 'slope', '.geo_y', 'bi', 'date', 'erc', 'eto', 'fm100',
       'fm1000', 'pr', 'rmax', 'rmin', 'th', 'tmmn', 'tmmx', 'vpd', 'vs',
       '.geo', 'pop_density', 'u_class'],
      dtype='str')

In [123]:
len(final_df)

2387

In [124]:
final_df.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2387 entries, 0 to 2386
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   SR_B2        2387 non-null   int64  
 1   SR_B3        2387 non-null   int64  
 2   SR_B4        2387 non-null   int64  
 3   SR_B5        2387 non-null   int64  
 4   SR_B7        2387 non-null   int64  
 5   date_x       2387 non-null   str    
 6   fire_id      2387 non-null   str    
 7   occur_id     2387 non-null   int64  
 8   point_id     2387 non-null   int64  
 9   .geo_x       2387 non-null   str    
 10  DEM          2387 non-null   float64
 11  aspect       2387 non-null   int64  
 12  date_y       2387 non-null   str    
 13  hillshade    2387 non-null   int64  
 14  slope        2387 non-null   int64  
 15  .geo_y       2387 non-null   str    
 16  bi           2387 non-null   float64
 17  date         2387 non-null   str    
 18  erc          2387 non-null   float64
 19  eto          2387

In [125]:
final_df['geometry'] = final_df['.geo'].fillna(final_df[".geo_x"]).fillna(final_df[".geo_y"])
final_df = final_df.drop(['.geo_x', '.geo_y', '.geo'], axis=1)

In [126]:
final_df.count()

SR_B2          2387
SR_B3          2387
SR_B4          2387
SR_B5          2387
SR_B7          2387
date_x         2387
fire_id        2387
occur_id       2387
point_id       2387
DEM            2387
aspect         2387
date_y         2387
hillshade      2387
slope          2387
bi             2387
date           2387
erc            2387
eto            2387
fm100          2387
fm1000         2387
pr             2387
rmax           2387
rmin           2387
th             2387
tmmn           2387
tmmx           2387
vpd            2387
vs             2387
pop_density    2387
u_class        2387
geometry       2387
dtype: int64

In [127]:
final_df

,SR_B2,SR_B3,SR_B4,SR_B5,SR_B7,date_x,fire_id,occur_id,point_id,DEM,...,rmax,rmin,th,tmmn,tmmx,vpd,vs,pop_density,u_class,geometry
0,8308,9251,9920,14829,12253,2025-01-29,VA3712108003520250129,9,0,531.721436,...,48.799999,20.4,276.0,274.700012,286.5,0.74,7.8,12.811959,11,"{""type"":""Point"",""coordinates"":[-80.04557007724..."
1,8660,9633,10132,14535,11009,2025-01-29,VA3712108003520250129,9,1,522.399597,...,48.799999,20.4,276.0,274.700012,286.5,0.74,7.8,12.811959,11,"{""type"":""Point"",""coordinates"":[-80.04539567321..."
2,9166,10703,11823,16475,13181,2025-01-29,VA3712108003520250129,9,2,522.399597,...,48.799999,20.4,276.0,274.700012,286.5,0.74,7.8,12.811959,11,"{""type"":""Point"",""coordinates"":[-80.04522126720..."
3,9100,10519,11630,16871,13964,2025-01-29,VA3712108003520250129,9,3,530.713928,...,48.799999,20.4,276.0,274.700012,286.5,0.74,7.8,12.811959,11,"{""type"":""Point"",""coordinates"":[-80.04504685921..."
4,8067,9184,10233,14435,14275,2025-01-29,VA3712108003520250129,9,4,530.713928,...,48.799999,20.4,276.0,274.700012,286.5,0.74,7.8,12.811959,11,"{""type"":""Point"",""coordinates"":[-80.04487244923..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2382,10177,11749,12803,14553,15376,2025-09-02,CA3622612010420250902,2,196,129.883575,...,53.200001,12.6,276.0,293.200012,313.0,3.67,2.7,0.000000,11,"{""type"":""Point"",""coordinates"":[-120.1486636823..."
2383,10210,11841,12912,14640,15479,2025-09-02,CA3622612010420250902,2,197,130.069687,...,53.200001,12.6,276.0,293.200012,313.0,3.67,2.7,0.000000,11,"{""type"":""Point"",""coordinates"":[-120.1489918330..."
2384,10294,11855,12961,14720,15491,2025-09-02,CA3622612010420250902,2,198,130.208099,...,53.200001,12.6,276.0,293.200012,313.0,3.67,2.7,0.000000,11,"{""type"":""Point"",""coordinates"":[-120.1493199806..."
2385,10148,11706,12752,14443,15346,2025-09-02,CA3622612010420250902,2,199,130.552017,...,53.200001,12.6,276.0,293.200012,313.0,3.67,2.7,0.000000,11,"{""type"":""Point"",""coordinates"":[-120.1496481251..."


In [128]:
final_df = final_df.drop(columns=['geometry'])

In [129]:
final_df.columns

Index(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'date_x', 'fire_id',
       'occur_id', 'point_id', 'DEM', 'aspect', 'date_y', 'hillshade', 'slope',
       'bi', 'date', 'erc', 'eto', 'fm100', 'fm1000', 'pr', 'rmax', 'rmin',
       'th', 'tmmn', 'tmmx', 'vpd', 'vs', 'pop_density', 'u_class'],
      dtype='str')

In [130]:
final_df.to_csv(r"C:\Users\jezkn\OneDrive\Documents\Birkbeck\Work\MSc Project\Wildfire Project\Outputs\VALIDATION_PartThreeOutput2.csv")